# Step 2: Baseline Models

This notebook starts from the cleaned LendingClub dataset produced in Step 1, prepares model-ready features, trains baseline Logistic Regression and XGBoost models, evaluates them, and saves the fitted pipelines for later explanation work.


## 1. Setup

Import packages and define shared configuration. Optional packages such as `optuna` and `xgboost` must be installed in the environment before running the tuning cells.


In [ ]:
from pathlib import Path
import pickle
import warnings
import optuna 

import joblib
import numpy as np
import pandas as pd
import statsmodels.api as sm

from xgboost import XGBClassifier

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    classification_report,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 80)

RANDOM_STATE = 42
TEST_SIZE = 0.30
CV_SPLITS = 5
N_OPTUNA_TRIALS = 20


## 2. Project paths

Use the Google Drive data folder in Colab when available. Otherwise, use a local `Data` folder inside the project directory.


In [ ]:
# PROJECT_ROOT = Path.cwd()
# COLAB_PROJECT_ROOT = Path("/content/drive/MyDrive/LLM_ClubLending")
# LOCAL_PROJECT_ROOT = PROJECT_ROOT

BASE_DIR = Path("/Users/yiqingwang/Documents/LLM_Exapliner")
DATA_DIR = BASE_DIR / "Data"
MODIFIED_DATA_DIR = DATA_DIR / "ModifiedData"
RESULT_DIR = BASE_DIR / "Result" / "BaselineModel"

CLEAN_DATA_PATH = MODIFIED_DATA_DIR / "CleanData.xlsx"
SPLIT_DATA_PATH = MODIFIED_DATA_DIR / "LC_split.pkl"
LOGISTIC_MODEL_PATH = RESULT_DIR / "logistic_pipeline_retrained.pkl"
XGB_MODEL_PATH = RESULT_DIR / "xgb_pipeline_retrained.pkl"

MODIFIED_DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Base directory: {BASE_DIR}")
print(f"Clean data path: {CLEAN_DATA_PATH}")
print(f"Result directory: {RESULT_DIR}")


## 3. Load cleaned data

Step 2 expects `CleanData.xlsx` from Step 1. If the file is missing, run `Step1_LoadData.ipynb` first.


In [ ]:
if not CLEAN_DATA_PATH.exists():
    raise FileNotFoundError(
        f"Clean data not found: {CLEAN_DATA_PATH}. Run Step1_LoadData.ipynb first."
    )

clean_df = pd.read_excel(CLEAN_DATA_PATH)

# Older versions of Step 1 saved the DataFrame index as an extra column.
if "Unnamed: 0" in clean_df.columns:
    clean_df = clean_df.drop(columns=["Unnamed: 0"])

print(f"Loaded clean data shape: {clean_df.shape}")
display(clean_df.head(3))


## 4. Additional feature cleaning

Convert percentage and term fields to numeric values, encode the binary target, remove free-text description fields, and reset the row index.


In [ ]:
def parse_percent_column(series):
    """Convert percentage values to decimals.

    Handles strings like '13.56%', whole-number percentages like 13.56,
    and values already stored as decimals like 0.1356.
    """
    as_text = series.astype(str).str.strip().replace({"nan": np.nan, "None": np.nan})
    has_percent_sign = as_text.str.contains("%", regex=False, na=False)
    numeric = as_text.str.replace("%", "", regex=False).astype(float)

    should_divide = has_percent_sign | (numeric > 1)
    return numeric.where(~should_divide, numeric / 100)


def parse_term_months(series):
    """Convert values like '36 months' into integer month counts."""
    return (
        series.astype(str)
        .str.replace("months", "", regex=False)
        .str.strip()
        .astype(int)
    )


clean_df = clean_df.replace("NaN", np.nan).copy()
clean_df["int_rate"] = parse_percent_column(clean_df["int_rate"])
clean_df["revol_util"] = parse_percent_column(clean_df["revol_util"])
clean_df["term"] = parse_term_months(clean_df["term"])
clean_df["loan_status"] = clean_df["loan_status"].replace({"Fully Paid": 0, "Charged Off": 1}).astype(int)

text_columns = ["desc", "title", "emp_title"]
existing_text_columns = [col for col in text_columns if col in clean_df.columns]
clean_df = clean_df.drop(columns=existing_text_columns)
clean_df = clean_df.reset_index(drop=True)

print(f"Dropped text columns: {existing_text_columns}")
print(clean_df["loan_status"].value_counts(dropna=False))


## 5. Region feature

Create a broad U.S. Census region feature from `addr_state`. This keeps location information while avoiding high-cardinality ZIP codes.


In [ ]:
STATE_TO_REGION = {
    "ME": "Northeast", "NH": "Northeast", "VT": "Northeast", "MA": "Northeast",
    "RI": "Northeast", "CT": "Northeast", "NY": "Northeast", "NJ": "Northeast", "PA": "Northeast",
    "OH": "Midwest", "IN": "Midwest", "IL": "Midwest", "MI": "Midwest", "WI": "Midwest",
    "MN": "Midwest", "IA": "Midwest", "MO": "Midwest", "ND": "Midwest", "SD": "Midwest",
    "NE": "Midwest", "KS": "Midwest",
    "DE": "South", "MD": "South", "DC": "South", "VA": "South", "WV": "South",
    "NC": "South", "SC": "South", "GA": "South", "FL": "South",
    "KY": "South", "TN": "South", "AL": "South", "MS": "South",
    "AR": "South", "LA": "South", "OK": "South", "TX": "South",
    "MT": "West", "ID": "West", "WY": "West", "CO": "West", "NM": "West",
    "AZ": "West", "UT": "West", "NV": "West", "CA": "West", "OR": "West", "WA": "West",
    "AK": "West", "HI": "West",
}

clean_df["zip3"] = clean_df["zip_code"].astype(str).str[:3]
clean_df["area"] = clean_df["addr_state"].map(STATE_TO_REGION).fillna("Other")

display(clean_df[["addr_state", "zip_code", "zip3", "area"]].head())
print(clean_df["area"].value_counts(dropna=False))


## 6. Missing values and categorical casting

Apply small, explicit imputations before the sklearn pipeline. Broad numerical/categorical missingness is still handled inside the preprocessing pipeline.


In [ ]:
missing_before = clean_df[["pub_rec_bankruptcies", "emp_length", "revol_util"]].isna().sum()

clean_df["pub_rec_bankruptcies"] = clean_df["pub_rec_bankruptcies"].fillna(0)
clean_df["emp_length"] = clean_df["emp_length"].fillna("Unknown")
clean_df["revol_util"] = clean_df["revol_util"].fillna(clean_df["revol_util"].mean())

clean_df["term"] = clean_df["term"].astype("category")
clean_df["pub_rec_bankruptcies"] = clean_df["pub_rec_bankruptcies"].astype(int).astype("category")

print("Missing values filled before pipeline:")
display(missing_before.rename("missing_count"))


## 7. Feature sets

Define the target, excluded identifiers/leakage fields, and the model feature groups used by the preprocessing pipeline.


In [ ]:
target = "loan_status"
excluded_features = [
    "loan_status",
    "id",
    "member_id",
    "zip3",
    "zip_code",
    "acc_now_delinq",
    "delinq_amnt",
    "tax_liens",
]

nominal_cat_features = [
    "home_ownership",
    "verification_status",
    "purpose",
    "area",
    "addr_state",
]

ordinal_cat_features = [
    "term",
    "grade",
    "sub_grade",
    "emp_length",
    "pub_rec_bankruptcies",
]

num_features = [
    "funded_amnt",
    "int_rate",
    "installment",
    "annual_inc",
    "dti",
    "delinq_2yrs",
    "fico_range_low",
    "fico_range_high",
    "inq_last_6mths",
    "open_acc",
    "pub_rec",
    "revol_bal",
    "revol_util",
    "total_acc",
]

selected_features = nominal_cat_features + ordinal_cat_features + num_features
missing_selected = [col for col in selected_features + [target] if col not in clean_df.columns]
if missing_selected:
    raise KeyError(f"Missing required columns: {missing_selected}")

unused_columns = [col for col in clean_df.columns if col not in selected_features + excluded_features]
print(f"Selected model features: {len(selected_features)}")
print(f"Unused columns not modeled in this baseline: {unused_columns}")


In [ ]:
len(selected_features), len(clean_df.columns), len(unused_columns)

In [ ]:
ordinal_categories = [
    [36, 60],
    ["A", "B", "C", "D", "E", "F", "G"],
    [
        "A1", "A2", "A3", "A4", "A5",
        "B1", "B2", "B3", "B4", "B5",
        "C1", "C2", "C3", "C4", "C5",
        "D1", "D2", "D3", "D4", "D5",
        "E1", "E2", "E3", "E4", "E5",
        "F1", "F2", "F3", "F4", "F5",
        "G1", "G2", "G3", "G4", "G5",
    ],
    [
        "Unknown", "< 1 year", "1 year", "2 years", "3 years", "4 years",
        "5 years", "6 years", "7 years", "8 years", "9 years", "10+ years",
    ],
    sorted(clean_df["pub_rec_bankruptcies"].dropna().astype(int).unique().tolist()),
]


## 8. Train/test split

Use a stratified split so the default rate is similar in training and testing data.


In [ ]:
# X = clean_df[selected_features].copy()
# y = clean_df[target].astype(int).copy()

# X_train, X_test, y_train, y_test = train_test_split(
#     X,
#     y,
#     test_size=TEST_SIZE,
#     stratify=y,
#     random_state=RANDOM_STATE,
# )

# with open(SPLIT_DATA_PATH, "wb") as f:
#     pickle.dump((X_train, X_test, y_train, y_test), f)

# print(f"Train shape: {X_train.shape}")
# print(f"Test shape: {X_test.shape}")
# print(f"Saved split data to: {SPLIT_DATA_PATH}")


In [ ]:
with open(SPLIT_DATA_PATH, "rb") as f:
    X_train, X_test, y_train, y_test = pickle.load(f)

X_train = X_train.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

## 9. Preprocessing pipeline

Numerical features use median imputation, nominal categorical features use one-hot encoding, and ordinal categorical features use ordered integer encoding.


In [ ]:
def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


numeric_transformer = SimpleImputer(strategy="median")

nominal_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", make_one_hot_encoder()),
    ]
)

ordinal_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "ordinal",
            OrdinalEncoder(
                categories=ordinal_categories,
                handle_unknown="use_encoded_value",
                unknown_value=-1,
            ),
        ),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_features),
        ("nominal", nominal_transformer, nominal_cat_features),
        ("ordinal", ordinal_transformer, ordinal_cat_features),
    ]
)


## 10. Evaluation helpers

These helpers keep test and train evaluation consistent across models.


In [ ]:
def calibration_metrics(y_true, y_prob, model_name="Model"):
    """Calculate probability calibration metrics for a fitted model.

    Brier Score measures overall probability error. Calibration intercept and
    slope are estimated by regressing the observed outcome on the logit of the
    model probability. A well-calibrated model has intercept near 0 and slope
    near 1.
    """
    eps = 1e-6
    y_prob = np.clip(np.asarray(y_prob), eps, 1 - eps)

    brier = brier_score_loss(y_true, y_prob)
    logit_p = np.log(y_prob / (1 - y_prob))
    X_calib = sm.add_constant(logit_p)
    calib_model = sm.Logit(y_true, X_calib).fit(disp=False)

    return {
        "Model": model_name,
        "Brier Score": brier,
        "Calibration Intercept": calib_model.params[0],
        "Calibration Slope": calib_model.params[1],
    }


def ks_table(y_true, y_prob, n_bins=10):
    """Return a decile KS table sorted from highest to lowest predicted risk."""
    data = pd.DataFrame({"target": y_true, "pred_prob": y_prob}).copy()
    data["non_event"] = 1 - data["target"]
    data["bucket"] = pd.qcut(data["pred_prob"], n_bins, duplicates="drop")

    grouped = data.groupby("bucket", as_index=False, observed=False)
    table = (
        data.groupby("bucket", observed=False)
        .agg(
            min_prob=("pred_prob", "min"),
            max_prob=("pred_prob", "max"),
            events=("target", "sum"),
            nonevents=("non_event", "sum")
        )
        .reset_index(drop=True)
    )

    table = table.sort_values("min_prob", ascending=False).reset_index(drop=True)
    table["event_rate"] = table["events"] / data["target"].sum()
    table["nonevent_rate"] = table["nonevents"] / data["non_event"].sum()
    table["cum_event_rate"] = table["event_rate"].cumsum()
    table["cum_nonevent_rate"] = table["nonevent_rate"].cumsum()
    table["KS"] = (table["cum_event_rate"] - table["cum_nonevent_rate"]).round(3) * 100
    table.index = range(1, len(table) + 1)
    table.index.name = "Decile"
    return table


def evaluate_pipeline(name, pipeline, X_eval, y_eval, threshold=0.5):
    y_prob = pipeline.predict_proba(X_eval)[:, 1]
    y_pred = (y_prob >= threshold).astype(int)

    print(f"{name} metrics")
    print(f"Accuracy: {accuracy_score(y_eval, y_pred):.4f}")
    print(f"ROC-AUC: {roc_auc_score(y_eval, y_prob):.4f}")
    print(f"PR-AUC: {average_precision_score(y_eval, y_prob):.4f}")
    print("Classification Report:")
    print(classification_report(y_eval, y_pred))

    calibration = calibration_metrics(y_eval, y_prob, model_name=name)
    display(pd.DataFrame([calibration]))

    deciles = ks_table(y_eval, y_prob)
    display(deciles)
    print(f"KS: {deciles['KS'].max():.1f}% at decile {deciles['KS'].idxmax()}")
    return {"pred": y_pred, "prob": y_prob, "ks": deciles, "calibration": calibration}


## 11. Logistic Regression baseline

Tune the regularization strength and class weighting using cross-validated PR-AUC, then fit the best pipeline on the full training split.


In [ ]:
import optuna


def logistic_objective(trial):
    model = LogisticRegression(
        penalty="l2",
        C=trial.suggest_float("C", 1e-2, 1e2, log=True),
        solver="lbfgs",
        max_iter=2000,
        class_weight=trial.suggest_categorical("class_weight", ["balanced", None]),
    )

    pipeline = Pipeline(steps=[("preprocess", preprocessor), ("model", model)])
    cv = StratifiedKFold(n_splits=CV_SPLITS, shuffle=True, random_state=RANDOM_STATE)

    scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring="average_precision",
        n_jobs=-1,
    )
    return scores.mean()


logistic_study = optuna.create_study(
    direction="maximize",
    study_name="logistic_pr_auc_tuning",
)
logistic_study.optimize(logistic_objective, n_trials=N_OPTUNA_TRIALS)

print(f"Best Logistic CV PR-AUC: {logistic_study.best_value:.4f}")
print(f"Best Logistic params: {logistic_study.best_params}")


In [ ]:
logistic_best_model = LogisticRegression(
    penalty="l2",
    C=logistic_study.best_params["C"],
    solver="lbfgs",
    max_iter=2000,
    class_weight=logistic_study.best_params.get("class_weight"),
)

logistic_pipeline = Pipeline(
    steps=[("preprocess", preprocessor), ("model", logistic_best_model)]
)
logistic_pipeline.fit(X_train, y_train)

logistic_test_results = evaluate_pipeline(
    "Logistic Regression test", logistic_pipeline, X_test, y_test
)

joblib.dump(logistic_pipeline, LOGISTIC_MODEL_PATH)
print(f"Saved Logistic pipeline to: {LOGISTIC_MODEL_PATH}")


## 12. XGBoost baseline

Tune an XGBoost classifier with class imbalance handled through `scale_pos_weight`.


In [ ]:
from xgboost import XGBClassifier

negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()
imbalance_ratio = negative_count / positive_count
print(f"Imbalance ratio, non-default/default: {imbalance_ratio:.3f}")


def create_xgb_model(trial):
    return XGBClassifier(
        n_estimators=trial.suggest_int("n_estimators", 200, 800),
        max_depth=trial.suggest_int("max_depth", 3, 8),
        learning_rate=trial.suggest_float("learning_rate", 0.02, 0.2, log=True),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0),
        min_child_weight=trial.suggest_float("min_child_weight", 1.0, 20.0, log=True),
        gamma=trial.suggest_float("gamma", 0.0, 5.0),
        reg_lambda=trial.suggest_float("reg_lambda", 1.0, 30.0, log=True),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        scale_pos_weight=trial.suggest_float(
            "scale_pos_weight",
            0.5 * imbalance_ratio,
            2.0 * imbalance_ratio,
            log=True,
        ),
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )


def xgb_objective(trial):
    pipeline = Pipeline(
        steps=[("preprocess", preprocessor), ("model", create_xgb_model(trial))]
    )
    cv = StratifiedKFold(n_splits=CV_SPLITS, shuffle=True, random_state=RANDOM_STATE)

    scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring="average_precision",
        n_jobs=-1,
    )
    return scores.mean()


xgb_study = optuna.create_study(
    direction="maximize",
    study_name="xgboost_pr_auc_tuning",
)
xgb_study.optimize(xgb_objective, n_trials=N_OPTUNA_TRIALS)

print(f"Best XGBoost CV PR-AUC: {xgb_study.best_value:.4f}")
print(f"Best XGBoost params: {xgb_study.best_params}")


In [ ]:
xgb_best_model = XGBClassifier(
    **xgb_study.best_params,
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    n_jobs=-1,
    random_state=RANDOM_STATE,
)

xgb_pipeline = Pipeline(steps=[("preprocess", preprocessor), ("model", xgb_best_model)])
xgb_pipeline.fit(X_train, y_train)

xgb_test_results = evaluate_pipeline("XGBoost test", xgb_pipeline, X_test, y_test)

joblib.dump(xgb_pipeline, XGB_MODEL_PATH)
print(f"Saved XGBoost pipeline to: {XGB_MODEL_PATH}")


## 13. Training-set performance check

This optional section reloads the saved split and saved models, then evaluates in-sample performance. Use it to check overfitting against the test metrics above.


In [ ]:
with open(SPLIT_DATA_PATH, "rb") as f:
    X_train_saved, X_test_saved, y_train_saved, y_test_saved = pickle.load(f)

logistic_saved_pipeline = joblib.load(LOGISTIC_MODEL_PATH)
xgb_saved_pipeline = joblib.load(XGB_MODEL_PATH)

logistic_train_results = evaluate_pipeline(
    "Logistic Regression train",
    logistic_saved_pipeline,
    X_train_saved,
    y_train_saved,
    threshold=0.556,
)

xgb_train_results = evaluate_pipeline(
    "XGBoost train",
    xgb_saved_pipeline,
    X_train_saved,
    y_train_saved,
    threshold=0.556,
)
